# XOR Gate — Naive MLP (fails) vs Backprop MLP (converges)

**Part A** — a naive multi-layer perceptron (step activation, plain delta rule, *no backprop*).
It gets stuck and never separates XOR — this is the expected failure.

**Part B** — a proper MLP: `2 -> 4 hidden (sigmoid) -> 1 (sigmoid)`, trained with
**backpropagation** + full-batch gradient descent. It learns a curved decision boundary and
separates XOR (0/4 misclassified).

Every decision boundary is plotted as its **own separate figure**. Both parts export into a
single PDF report.

In [ ]:
!apt-get install -y fonts-liberation -q > /dev/null 2>&1
!pip install reportlab pdf2image -q
!apt-get install -y poppler-utils -q > /dev/null 2>&1

In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib as mpl, os
mpl.rcParams["font.family"]="serif"
mpl.rcParams["font.serif"]=["Liberation Serif","Times New Roman","DejaVu Serif"]
mpl.rcParams["font.size"]=13
mpl.rcParams["savefig.dpi"]=600; mpl.rcParams["figure.dpi"]=120
os.makedirs("figs",exist_ok=True); os.makedirs("figs2",exist_ok=True)
X=np.array([[0,0],[0,1],[1,0],[1,1]],dtype=float)

## Part A — Naive MLP (no backprop)

In [ ]:
np.random.seed(42)
y_step=np.array([0,1,1,0])
class NaiveMLP:
    def __init__(self,ni,nh,lr=0.1):
        self.lr=lr
        self.W_hidden=np.random.uniform(-0.1,0.1,(nh,ni))
        self.b_hidden=np.random.uniform(-0.1,0.1,nh)
        self.W_out=np.random.uniform(-0.1,0.1,nh)
        self.b_out=np.random.uniform(-0.1,0.1); self.history=[]
    @staticmethod
    def step(z): return np.where(z>=0,1,0)
    def forward(self,x):
        ha=self.step(self.W_hidden@x+self.b_hidden); return ha,self.step(self.W_out@ha+self.b_out)
    def snap(self): self.history.append((self.W_hidden.copy(),self.b_hidden.copy(),self.W_out.copy(),self.b_out))
    def fit(self,X,y,epochs=30):
        self.snap(); self.epe=[]
        for e in range(1,epochs+1):
            err=0
            for xi,t in zip(X,y):
                ha,pred=self.forward(xi); d=t-pred
                if d!=0:
                    self.W_out+=self.lr*d*ha; self.b_out+=self.lr*d
                    self.W_hidden+=self.lr*d*np.outer(np.ones_like(self.b_hidden),xi)
                    self.b_hidden+=self.lr*d*np.ones_like(self.b_hidden); err+=1; self.snap()
                    print(f"Epoch {e}, sample {xi}, target {t}, pred {pred} -> W_hidden={np.round(self.W_hidden,3).tolist()}, b_hidden={np.round(self.b_hidden,3).tolist()}, W_out={np.round(self.W_out,3)}, b_out={self.b_out:.3f}")
            self.epe.append(err); print(f"  epoch {e} misclassified: {err}")
            if err==0: break
        return self.history
mlp=NaiveMLP(2,2,0.1); hist1=mlp.fit(X,y_step,30)
print("Misclassified per epoch:",mlp.epe)

In [ ]:
# Separate boundary figure per update (Part A)
xx,yy=np.meshgrid(np.linspace(-0.5,1.5,200),np.linspace(-0.5,1.5,200)); grid=np.c_[xx.ravel(),yy.ravel()]
bd1=[]
for i,(Wh,bh,Wo,bo) in enumerate(hist1[:16]):
    pr=np.array([1 if (Wo@np.where(Wh@p+bh>=0,1,0)+bo)>=0 else 0 for p in grid]).reshape(xx.shape)
    fig,ax=plt.subplots(figsize=(4.2,4.2))
    ax.contourf(xx,yy,pr,levels=[-0.5,0.5,1.5],colors=["#f7c9c2","#bfe3c8"],alpha=0.85)
    for xi,t in zip(X,y_step): ax.scatter(xi[0],xi[1],c=("seagreen" if t==1 else "crimson"),s=130,edgecolor="black",zorder=3)
    ax.set_title("Initial" if i==0 else f"Update {i}"); ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_xlim(-0.5,1.5); ax.set_ylim(-0.5,1.5); plt.tight_layout()
    f=f"figs/bd_{i:02d}.pdf"; plt.savefig(f); bd1.append(f); plt.show()
plt.figure(figsize=(7,5)); plt.plot(range(1,len(mlp.epe)+1),mlp.epe,marker="o",color="darkorange")
plt.xlabel("Epoch"); plt.ylabel("Misclassified Samples"); plt.title("Naive MLP on XOR: Training Error vs Epoch")
plt.tight_layout(); plt.savefig("figs/error_curve.pdf"); plt.show()

## Part B — Backprop MLP (`2 -> 4 sigmoid -> 1 sigmoid`)

Full-batch gradient descent, eta=0.5, 5000 epochs, weights ~ Uniform(-1,1) with a fixed seed.
Boundary snapshotted every 200 epochs.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))
Y=np.array([[0],[1],[1],[0]],dtype=float)
rng=np.random.RandomState(0)
W1=rng.uniform(-1,1,(2,4)); b1=rng.uniform(-1,1,(1,4))
W2=rng.uniform(-1,1,(4,1)); b2=rng.uniform(-1,1,(1,1))
lr=0.5; epochs=5000; snap=200
def ev(W1,b1,W2,b2):
    a1=sigmoid(X@W1+b1); yh=sigmoid(a1@W2+b2)
    return np.mean((Y-yh)**2), int(np.sum((yh>=0.5).astype(int)!=Y.astype(int)))
hist2=[]; mse=[]; mis=[]; norms=[]
m,mi=ev(W1,b1,W2,b2); hist2.append((0,W1.copy(),b1.copy(),W2.copy(),b2.copy()))
for ep in range(1,epochs+1):
    a1=sigmoid(X@W1+b1); yh=sigmoid(a1@W2+b2); e=Y-yh
    d2=e*yh*(1-yh); dW2=a1.T@d2; db2=d2.sum(0,keepdims=True)
    d1=(d2@W2.T)*a1*(1-a1); dW1=X.T@d1; db1=d1.sum(0,keepdims=True)
    W2+=lr*dW2; b2+=lr*db2; W1+=lr*dW1; b1+=lr*db1
    mm,mmi=ev(W1,b1,W2,b2); mse.append(mm); mis.append(mmi)
    if ep%snap==0:
        hist2.append((ep,W1.copy(),b1.copy(),W2.copy(),b2.copy()))
        norms.append((ep,np.linalg.norm(W1),np.linalg.norm(b1),np.linalg.norm(W2),np.linalg.norm(b2)))
first0=next((i+1 for i,v in enumerate(mis) if v==0),None)
print(f"Final MSE={mse[-1]:.6f}, final misclassified={mis[-1]}/4, first 0-mis at epoch {first0}")

In [ ]:
# Separate boundary figure per logged update (Part B)
bd2=[]
for (ep,w1,bb1,w2,bb2) in hist2:
    a1=sigmoid(grid@w1+bb1); pr=sigmoid(a1@w2+bb2).reshape(xx.shape)
    fig,ax=plt.subplots(figsize=(4.2,4.2))
    ax.contourf(xx,yy,pr,levels=[0,0.5,1],colors=["#f7c9c2","#bfe3c8"],alpha=0.85)
    ax.contour(xx,yy,pr,levels=[0.5],colors="black",linewidths=0.8)
    for xi,t in zip(X,Y.ravel()): ax.scatter(xi[0],xi[1],c=("seagreen" if t==1 else "crimson"),s=130,edgecolor="black",zorder=3)
    ax.set_title(f"Epoch {ep}"); ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_xlim(-0.5,1.5); ax.set_ylim(-0.5,1.5); plt.tight_layout()
    f=f"figs2/bd_{ep:04d}.pdf"; plt.savefig(f); bd2.append(f); plt.show()

In [ ]:
# Loss, convergence, weight-norm curves (Part B)
plt.figure(figsize=(7,5)); plt.plot(range(1,epochs+1),mse,color="crimson")
plt.xlabel("Epoch"); plt.ylabel("Mean Squared Error"); plt.title("MLP Trained on XOR -- Loss Curve")
plt.tight_layout(); plt.savefig("figs2/loss.pdf"); plt.show()
plt.figure(figsize=(7,5)); plt.plot(range(1,epochs+1),mis,color="seagreen")
plt.xlabel("Epoch"); plt.ylabel("Misclassified Samples (out of 4)"); plt.title("MLP Trained on XOR -- Convergence")
plt.ylim(-0.3,4.3); plt.tight_layout(); plt.savefig("figs2/convergence.pdf"); plt.show()
na=np.array(norms)
plt.figure(figsize=(7,5))
for k,lbl in zip([1,2,3,4],["||W1|| (input -> hidden)","||b1|| (hidden bias)","||W2|| (hidden -> output)","||b2|| (output bias)"]):
    plt.plot(na[:,0],na[:,k],marker="o",ms=3,label=lbl)
plt.xlabel("Epoch (logged update)"); plt.ylabel("L2 Norm"); plt.title("MLP Weight Evolution"); plt.legend(fontsize=9)
plt.tight_layout(); plt.savefig("figs2/wnorm.pdf"); plt.show()

## Analysis

**Part A (naive rule) fails:** every hidden neuron gets the same output error (no per-neuron
credit assignment) and the step activation has zero gradient, so the hidden layer can't learn a
useful representation — the error freezes at 2/4 and the boundary never separates XOR.

**Part B (backprop) works:** each hidden neuron learns its own sigmoid boundary; backprop
assigns each its own error term via the chain rule, and the differentiable sigmoid lets gradients
flow. Composing 4 linear boundaries yields the curved non-linear region that places all four XOR
points on their correct sides — MSE decays to ~0 and misclassifications hit 0/4 and stay there.

## Export the combined PDF report

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import SimpleDocTemplate,Paragraph,Spacer,Image,Table,TableStyle,PageBreak
from reportlab.lib import colors
from pdf2image import convert_from_path
import os
os.makedirs("png_all",exist_ok=True)
def topng(p):
    o="png_all/"+os.path.basename(p).replace(".pdf",".png"); convert_from_path(p,dpi=200)[0].save(o); return o
P1=[topng(f) for f in bd1]; E1=topng("figs/error_curve.pdf")
P2=[topng(f) for f in bd2]; L2=topng("figs2/loss.pdf"); C2=topng("figs2/convergence.pdf"); WN=topng("figs2/wnorm.pdf")
body=ParagraphStyle("b",fontName="Times-Roman",fontSize=11,leading=15,alignment=4)
h1=ParagraphStyle("h1",fontName="Times-Bold",fontSize=15); h2=ParagraphStyle("h2",fontName="Times-Bold",fontSize=12.5,spaceBefore=6)
cap=ParagraphStyle("c",parent=body,fontSize=9.5,alignment=1,leading=12)
def gridof(pngs,labels):
    IW=7.4*cm; flat=[]
    for i in range(0,len(pngs),2):
        cs=[]
        for j in (i,i+1):
            if j<len(pngs):
                t=Table([[Image(pngs[j],width=IW,height=IW)],[Paragraph(labels[j],cap)]]); t.setStyle(TableStyle([("ALIGN",(0,0),(-1,-1),"CENTER")])); cs.append(t)
            else: cs.append("")
        flat.append(cs)
    g=Table(flat,colWidths=[8.2*cm,8.2*cm]); g.setStyle(TableStyle([("ALIGN",(0,0),(-1,-1),"CENTER"),("VALIGN",(0,0),(-1,-1),"TOP"),("BOTTOMPADDING",(0,0),(-1,-1),10)])); return g
doc=SimpleDocTemplate("XOR_MLP_Report.pdf",pagesize=A4,leftMargin=2*cm,rightMargin=2*cm,topMargin=1.8*cm,bottomMargin=1.8*cm)
E=[Paragraph("Additional Task (Part A)",h1),Paragraph("Naive MLP for XOR (No Backpropagation)",h2),
   Paragraph("2 inputs -> 2 hidden (step) -> 1 output (step), trained with the plain delta rule, no backprop. Each decision-boundary update is a separate figure below.",body),Spacer(1,6),
   gridof(P1,["Initial" if i==0 else f"Update {i}" for i in range(len(P1))]),
   Paragraph("Figure 1: Naive MLP - decision boundary after each update (never separates XOR).",cap),Spacer(1,8),
   Image(E1,width=11*cm,height=7.85*cm),Paragraph("Figure 2: Naive MLP - training error stuck at 2/4.",cap),PageBreak(),
   Paragraph("Additional Task (Part B)",h1),Paragraph("Backprop MLP for XOR (2 -> 4 sigmoid -> 1 sigmoid)",h2),
   Paragraph(f"Full-batch gradient descent, eta=0.5, 5000 epochs, Uniform(-1,1) init (RandomState(0)). Final MSE {mse[-1]:.6f}, {mis[-1]}/4 misclassified; first reaches 0/4 near epoch {first0}. Each logged boundary is a separate figure.",body),Spacer(1,6),
   gridof(P2,[f"Epoch {ep}" for ep,_,_,_,_ in hist2]),
   Paragraph("Figure 3: Backprop MLP - boundary bends into a curved region separating XOR.",cap),PageBreak(),
   Image(C2,width=11*cm,height=7.85*cm),Paragraph("Figure 4: Convergence - misclassified drops to 0/4 and stays.",cap),Spacer(1,6),
   Image(L2,width=11*cm,height=7.85*cm),Paragraph("Figure 5: Loss curve - MSE decays smoothly to ~0.",cap),Spacer(1,6),
   Image(WN,width=11*cm,height=7.85*cm),Paragraph("Figure 6: Weight L2 norms settle into a stable set.",cap),Spacer(1,8),
   Paragraph("Analysis: the naive rule gives every hidden neuron the same error and uses a zero-gradient step activation, so it never learns; backprop assigns each hidden neuron its own error via the chain rule and the sigmoid lets gradients flow, composing several linear boundaries into the non-linear region XOR requires.",body)]
doc.build(E); print("Saved XOR_MLP_Report.pdf")
# Colab download:
# from google.colab import files; files.download("XOR_MLP_Report.pdf")